# GNN MAPPO (based on Li. et al)

### We use a modified simple adversarial environment ported from PettingZoo MPE

### Graph Building and Norm

In [ ]:
import torch
import typing
from vmas.simulator.core import Agent, World, Landmark, Sphere
from vmas.simulator.scenario import BaseScenario
from vmas.simulator.utils import Color, ScenarioUtils

# Agent Type definitions
SCOUT = "scout"
INTERCEPTOR = "interceptor"
INTRUDER = "intruder"


class Scenario(BaseScenario):
    """
    Guarded Territory: heterogenus cooperative-competitive MARL scenario

    Specifically designed for our GNN use case with
    - Heterogenus observation spaces (scouts see further, interceptors see closer)
    - Communication contraints (GNN is the only communication channel between agents)
    - Mixed cooperative-competitive dynamics (defenters cooperate, intruders compete)

    Key Principle: Interceptor CANNOT succeed without scout communication
    Ensures that r_comm and K ablations produce strong, interpretable signals
    """

    def make_world(self, batch_dim: int, device: torch.device, **kwargs):
        # Agent / World Config
        self.n_scouts = kwargs.get("n_scouts", 3)
        self.n_interceptors = kwargs.get("n_interceptors", 3)
        self.n_intruders = kwargs.get("n_intruders", 3)
        self.n_zones = kwargs.get("n_zones", 2)
        self.world_size = kwargs.get("world_size", 5.0)

        # Observation Radii - core asymmetry
        self.scout_fov = kwargs.get("scout_fov", 1.0)
        self.interceptor_fov = kwargs.get("interceptor_fov", 0.5)
        self.tag_radius = kwargs.get("tag_radius", 0.1)

        # Speeds
        self.intruder_speed = kwargs.get("intruder_speed", 0.5)
        self.defender_speed = kwargs.get("defender_speed", 0.8)

        self.n_defenders = self.n_scouts + self.n_interceptors

        # Use Scripted Intruder (default)
        self.intruder_script = kwargs.get("scripted_intruder", True)

        # Create World
        world = World(
            batch_dim=batch_dim,
            device=device,
            dt=0.1,
            drag=0.25,
            dim_c=0, # no build in comms
            x_semidim=self.world_size,
            y_semidim=self.world_size
        )

        # Create Scouts
        self.scouts = []
        for i in range(self.n_scouts):
            agent = Agent(
                name=f"scout_{i}",
                collide=True,
                mass=1.0,
                shape=Sphere(radius=0.075),
                max_speed=self.defender_speed,
                color=Color.BLUE,
                u_range=1.0
            )
            agent.agent_type = SCOUT
            agent.type_id = 0
            world.add_agent(agent=agent)
            self.scouts.append(agent)


        # Create Interceptors
        self.interceptors = []
        for i in range(self.n_interceptors):
            agent = Agent(
                name=f"interceptor_{i}",
                collide=True,
                mass=1.0,
                shape=Sphere(radius=0.09),
                max_speed=self.defender_speed,
                color=Color.GREEN,
                u_range=1.0
            )
            agent.agent_type = INTERCEPTOR
            agent.type_id = 1
            world.add_agent(agent)
            self.interceptors.append(agent)

        # Create Interuders (Scripted)
        self.intruders = []
        for i in range(self.n_intruders):
            intruder = Agent(
                name=f"intruder_{i}",
                collide=True,
                mass=1.0,
                shape=Sphere(radius=0.075),
                max_speed=self.intruder_speed,
                color=Color.RED,
                u_range=1.0,
                )
            intruder.agent_type = INTRUDER
            intruder.type_id = 2
            world.add_agent(intruder)
            self.intruders.append(intruder)


        self.defenders = self.scouts + self.interceptors

        # Create Target Zones / Landmarks
        self.zones = []
        for i in range(self.n_zones):
            zone = Landmark(
                name=f"zone_{i}",
                collide=False,
                movable=False,
                shape=Sphere(radius=0.2),
                color=Color.LIGHT_GREEN
            )
            world.add_landmark(zone)
            self.zones.append(zone)


        # Tracking Tensors (allocated in reset)
        self._intruder_tagged = None
        self._zone_breached = None
        self._tag_count = None

        return world
    

    def reset_world_at(self, env_index: typing.Optional[int] = None):
        """
        Spawns agents and landmarks during env.reset()

        Layout:
        - Zones placed in the inner region
        - Defenders spawn near zones
        - Intruders spawn on the outer edges
        """
        batch = self.world.batch_dim
        device = self.world.device

        if env_index is None:
            self._intruder_tagged = torch.zeros(
                batch, self.n_intruders, dtype=torch.bool, device=device
            )
            self._zone_breached = torch.zeros(
                batch, self.n_zones, dtype=torch.bool, device=device
            )
            self._tag_count = torch.zeros(batch, dtype=torch.float32, device=device)
        else:
            self._intruder_tagged[env_index] = False
            self._zone_breached[env_index] = False
            self._tag_count[env_index] = 0.0
        
        # spawn zones in inner region
        for i, zone in enumerate(self.zones):
            pos = torch.zeros(
                (1,2) if env_index is not None else (batch, 2),
                dtype=torch.float32,
                device=device
            )

            # spread zones along x-axis near center
            angle = 2 * torch.pi * i / self.n_zones
            radius = 0.3
            pos[..., 0] = radius * torch.cos(torch.tensor(angle))
            pos[...,1] = radius * torch.sin(torch.tensor(angle))

            # add small noise
            pos += 0.1 * torch.randn_like(pos)
            zone.set_pos(pos, batch_index=env_index)

        for i, defender in enumerate(self.defenders):
            pos = torch.zeros(
                (1,2) if env_index is not None else (batch,2),
                dtype=torch.float32,
                device=device
            )
            angle = 2 * torch.pi * i / self.n_defenders
            radius = 0.5 + 0.2 * torch.rand(pos.shape[0], 1, device=device)
            pos[...,0:1] = radius * torch.cos(torch.tensor(angle))
            pos[...,1:2] = radius * torch.sin(torch.tensor(angle))
            pos += 0.05 * torch.randn_like(pos)
            defender.set_pos(pos, batch_index=env_index)

        # spawn intruders on outer edge
        for i, intruder in enumerate(self.intruders):
            pos = torch.zeros(
                (1, 2) if env_index is not None else (batch, 2),
                dtype=torch.float32,
                device=device,
            )
            angle = 2 * torch.pi * i / self.n_intruders + torch.pi  # opposite side
            radius = self.world_size * 0.85
            pos[..., 0] = radius * torch.cos(torch.tensor(angle))
            pos[..., 1] = radius * torch.sin(torch.tensor(angle))
            pos += 0.1 * torch.randn_like(pos)
            intruder.set_pos(pos, batch_index=env_index)

    def _get_intruder_actions(self, intruder: Agent) -> torch.Tensor:
        """
        Compute scripted intruder action: navigate to nearest zone with noise.
        Returns action tensor of shape (batch, 2).
        """
        device = self.world.device
        batch = self.world.batch_dim

        # Find nearest zone
        min_dist = torch.full((batch,), float("inf"), device=device)
        target_pos = self.zones[0].state.pos.clone()

        for zone in self.zones:
            dist = torch.linalg.vector_norm(
                intruder.state.pos - zone.state.pos, dim=-1
            )
            closer = dist < min_dist
            min_dist = torch.where(closer, dist, min_dist)
            target_pos = torch.where(closer.unsqueeze(-1), zone.state.pos, target_pos)

        # Direction toward target + exploration noise
        direction = target_pos - intruder.state.pos
        direction = direction / (torch.linalg.vector_norm(direction, dim=-1, keepdim=True) + 1e-6)
        noise = 0.2 * torch.randn(batch, 2, device=device)
        action = self.intruder_speed * (direction + noise)

        return action
    
    def process_action(self, agent: Agent):
        """
        Override to inject scripted actions for intruders.
        For defenders, actions come from the learned policy (no-op here).
        """
        if hasattr(agent, "agent_type") and agent.agent_type == INTRUDER:
            action = self._get_intruder_actions(agent)
            agent.action.u = action


    def observation(self, agent: Agent) -> torch.Tensor:
        """
        Build obs vector for agent 
        Differs by agent type
        
        Scouts (large FOV):
            [own_vel(2), own_pos(2), type_one_hot(2),
             zone_rel_pos(n_zones*2),
             visible_intruders_rel_pos(n_intruders*2) <- sees more
             visible_intruders_vel(n_intruders*2)
             nearby_defenders_rel_pos(n_defenders-1)*2] 

        Interceptors (small FOV):
            [own_vel(2), own_pos(2), type_one_hot(2),
             zone_rel_pos(n_zones*2),
             visible_intruders_rel_pos(n_intruders*2) <- sees less (masked)
             visible_intruders_vel(n_intruders*2),
             nearby_defenders_rel_pos((n_defenders-1)*2)]

        CRITICAL: Intruder visibility is masked by FOV radius.
        Agents outside FOV get zero-filled observations.
        This is what makes GNN communication essential — scouts see
        intruders that interceptors cannot, and must relay this info.

        NOTE: We make obs dim identical across agent types by zero masking out of range entities
        """
        batch = self.world.batch_dim
        device = self.world.device

        if hasattr(agent, "agent_type") and agent.agent_type == SCOUT:
            fov = self.scout_fov
            type_oh = torch.tensor([1.0,0.0], device=device).expand(batch, 2)
        elif hasattr(agent, "agent_type") and agent.agent_type == INTERCEPTOR:
            fov = self.interceptor_fov
            type_oh = torch.tensor([0.0,1.0], device=device).expand(batch,2)
        else:
            # intruders get dubby obs since scripted
            return torch.zeros(batch,2,device=device) 
        
        obs_parts = []
        obs_parts.append(agent.state.vel) # (batch,2)
        obs_parts.append(agent.state.pos) # (batch,2)
        obs_parts.append(type_oh)         # (batch,2)

        # zone relative positions (always visible)
        for zone in self.zones:
            obs_parts.append(zone.state.pos - agent.state.pos)  # (batch,2)

        # intruder obs (fov masked)
        for intruder in self.intruders:
            rel_pos = intruder.state.pos - agent.state.pos
            dist = torch.linalg.vector_norm(rel_pos, dim=-1, keepdim=True) 
            visible = (dist <= fov).float()

            obs_parts.append(rel_pos * visible)
            obs_parts.append(intruder.state.vel * visible)
        
        # other defender rel pos (fov masked)
        for other in self.defenders:
            if other is agent:
                continue
            rel_pos = other.state.pos - agent.state.pos
            dist = torch.linalg.vector_norm(rel_pos, dim=-1, keepdim=True)
            visible = (dist <= fov).float()

            obs_parts.append(rel_pos*visible)

        return torch.cat(obs_parts, dim=-1)


    def reward(self, agent: Agent) -> torch.Tensor:
        """
        Compute reward for the given agent. Return shape (batch,)

        intruders are scripted so return 0s
        Defender reward structure (shared team reward + individual shaping):

        1. TEAM REWARD (same for all defenders — enables CTDE):
           - Large penalty when intruder reaches a zone (zone breached)
           - Large bonus when interceptor tags an intruder

        2. INDIVIDUAL SHAPING (differs by type):
           Scouts:
             - Small reward for being near intruders (encourages scouting)
             - Small reward for being within comm range of interceptors
               (encourages information relay positioning)
           Interceptors:
             - Distance-based shaping toward nearest untagged intruder
             - Bonus for successful tags

        This reward design means scouts are incentivized to position
        themselves as communication bridges, not just passive observers.
        """

        if hasattr(agent, "agent_type") and agent.agent_type == INTRUDER:
            return torch.zeros(self.world.batch_dim, device=self.world.device)
        
        batch = self.world.batch_dim
        device = self.world.device
        rew = torch.zeros(batch, device=device)

        # compute tagging events
        for j, intruder in enumerate(self.intruders):
            if self._intruder_tagged is None:
                break
            already_tagged = self._intruder_tagged[:, j]

            for interceptor in self.interceptors:
                dist = torch.linalg.vector_norm(
                    interceptor.state.pos - intruder.state.pos, dim=-1
                )

                just_tagged = (~already_tagged) & (dist < self.tag_radius)
                self._intruder_tagged[:,j] = self._intruder_tagged[:, j] | just_tagged

                self._tag_count += just_tagged.float()


        # compute zone breach events
        for k, zone in enumerate(self.zones):
            for j, intruder in enumerate(self.intruders):
                if self._intruder_tagged is None:
                    break
                tagged = self._intruder_tagged[:, j]
                dist_to_zone = torch.linalg.vector_norm(
                    intruder.state.pos - zone.state.pos, dim=-1
                )
                breached = (~tagged) & (dist_to_zone < 0.15)
                # only count first breach per zone per ep
                new_breached = breached & (~self._zone_breached[:,k])
                self._zone_breached[:, k] = self._zone_breached[:,k] | breached

                rew -= 5.0 * new_breached.float()



        # team bonus: per-step differential, reward the tag in the step it happens
        for j, intruder in enumerate(self.intruders):
            for interceptor in self.interceptors:
                dist = torch.linalg.vector_norm(
                    interceptor.state.pos - intruder.state.pos, dim=-1
                )
                tagged = self._intruder_tagged[:, j]

                #give team bonus only on the step the tag occurs
                close_to_untagged = (~tagged) & (dist < self.tag_radius)
                rew += 3.0 * close_to_untagged.float()


        # individual shaping
        if hasattr(agent, "agent_type") and agent.agent_type == SCOUT:
            # Scouts: reward proximity to intruders (scouting)
            for intruder in self.intruders:
                dist = torch.linalg.vector_norm(
                    agent.state.pos - intruder.state.pos, dim=-1
                )
                # Reward for keeping intruders in FOV
                in_fov = (dist < self.scout_fov).float()
                rew += 0.1 * in_fov

            # Scouts: reward being within potential comm range of interceptors
            # This encourages relay positioning behavior
            for interceptor in self.interceptors:
                dist = torch.linalg.vector_norm(
                    agent.state.pos - interceptor.state.pos, dim=-1
                )
                # Reward for being at a "useful" relay distance
                # Not too close (wasting the FOV advantage), not too far
                good_relay_dist = (dist > 0.2) & (dist < 1.0)
                rew += 0.05 * good_relay_dist.float()

        elif hasattr(agent, "agent_type") and agent.agent_type == INTERCEPTOR:
            # Interceptors: shaped reward toward nearest untagged intruder
            min_dist = torch.full((batch,), float("inf"), device=device)
            for j, intruder in enumerate(self.intruders):
                if self._intruder_tagged is not None:
                    tagged = self._intruder_tagged[:, j]
                else:
                    tagged = torch.zeros(batch, dtype=torch.bool, device=device)

                dist = torch.linalg.vector_norm(
                    agent.state.pos - intruder.state.pos, dim=-1
                )
                # Only consider untagged intruders
                effective_dist = torch.where(tagged, torch.tensor(float("inf"), device=device), dist)
                min_dist = torch.minimum(min_dist, effective_dist)

            # Negative distance shaping (closer = better)
            # Clamp to avoid inf when all intruders tagged
            min_dist = torch.clamp(min_dist, max=5.0)
            rew -= 0.1 * min_dist

        return rew
    

    # done condition
    def done(self) -> torch.Tensor:
        """
        Episode ends when:
        - All intruders are tagged (defenders win), OR
        - All zones are breached (intruders win)

        Returns (batch,) bool tensor.
        """
        if self._intruder_tagged is None or self._zone_breached is None:
            return torch.zeros(
                self.world.batch_dim, dtype=torch.bool, device=self.world.device
            )

        all_tagged = self._intruder_tagged.all(dim=-1)   # (batch,)
        all_breached = self._zone_breached.all(dim=-1)   # (batch,)
        return all_tagged | all_breached
    
    def info(self, agent: Agent) -> dict:
        """Return extra info for logging during training."""
        info = {}
        if self._intruder_tagged is not None:
            info["n_tagged"] = self._intruder_tagged.sum(dim=-1).float()
        if self._zone_breached is not None:
            info["n_breached"] = self._zone_breached.sum(dim=-1).float()
        return info




def get_obs_dim(n_scouts=3, n_interceptors=3, n_intruders=3, n_zones=2):
    """
    Compute the observation dimension for defenders.
    Useful for constructing your GNN-MAPPO policy networks.

    obs = [vel(2) + pos(2) + type_oh(2)
        + zones(n_zones*2)
        + intruders_rel_pos(n_intruders*2) + intruders_vel(n_intruders*2)
        + other_defenders_rel_pos((n_scouts+n_interceptors-1)*2)]
    """
    n_defenders = n_scouts + n_interceptors
    obs_dim = (
        2       # vel
        + 2     # pos
        + 2     # type one-hot
        + n_zones * 2              # zone relative positions
        + n_intruders * 2          # intruder relative positions (masked)
        + n_intruders * 2          # intruder velocities (masked)
        + (n_defenders - 1) * 2    # other defenders relative positions
    )
    return obs_dim



### VMAS-Petting Zoo Adapter for easier integration

In [ ]:
class GuardedTerritoryAdapter:
    """
    Wraps the VMAS Guarded Territory scenario to produce outputs
    compatible with your GNN-MAPPO training loop.

    The adapter:
    - Filters out intruder agents (scripted, not learned)
    - Stacks defender observations into (num_envs, n_defenders, obs_dim)
    - Provides positions for adjacency matrix construction
    - Returns a single team reward (shared across defenders for CTDE)
    - Handles the global done flag (broadcasts to all defenders)
    """

    def __init__(
        self,
        num_envs: int = 64,
        device: str = "cpu",
        n_scouts: int = 3,
        n_interceptors: int = 3,
        n_intruders: int = 3,
        n_zones: int = 2,
        max_steps: int = 200,
        **kwargs,
    ):
        self.num_envs = num_envs
        self.device = device
        self.n_scouts = n_scouts
        self.n_interceptors = n_interceptors
        self.n_intruders = n_intruders
        self.n_defenders = n_scouts + n_interceptors
        self.n_zones = n_zones

        # Create the VMAS environment
        self.env = vmas.make_env(
            scenario=Scenario(),
            num_envs=num_envs,
            device=device,
            continuous_actions=True,
            max_steps=max_steps,
            n_scouts=n_scouts,
            n_interceptors=n_interceptors,
            n_intruders=n_intruders,
            n_zones=n_zones,
            **kwargs,
        )

        # Compute obs dim for network construction
        self.obs_dim = get_obs_dim(n_scouts, n_interceptors, n_intruders, n_zones)

        # Identify which indices in env.agents are defenders vs intruders
        self.defender_indices = []
        self.intruder_indices = []
        for i, agent in enumerate(self.env.agents):
            if hasattr(agent, "agent_type"):
                if agent.agent_type in (SCOUT, INTERCEPTOR):
                    self.defender_indices.append(i)
                elif agent.agent_type == INTRUDER:
                    self.intruder_indices.append(i)

        assert len(self.defender_indices) == self.n_defenders, (
            f"Expected {self.n_defenders} defenders, found {len(self.defender_indices)}"
        )

        # Store agent type info for potential type-conditioned processing
        self.agent_types = []  # 0=scout, 1=interceptor
        for idx in self.defender_indices:
            agent = self.env.agents[idx]
            self.agent_types.append(agent.type_id)
        self.agent_types = torch.tensor(self.agent_types, device=device)

    def reset(self):
        """
        Reset the environment.

        Returns:
            obs: (num_envs, n_defenders, obs_dim) — stacked defender observations
            positions: (num_envs, n_defenders, 2) — defender positions for adj matrix
        """
        all_obs = self.env.reset()  # tuple of (num_envs, obs_dim_i) tensors

        # Stack defender observations
        defender_obs = torch.stack(
            [all_obs[i] for i in self.defender_indices], dim=1
        )  # (num_envs, n_defenders, obs_dim)

        # Extract positions from observations (pos is at indices 2:4 in obs)
        positions = defender_obs[:, :, 2:4].clone()  # (num_envs, n_defenders, 2)

        return defender_obs, positions

    def step(self, defender_actions: torch.Tensor):
        """
        Step the environment with learned defender actions.
        Intruder actions are handled internally by the scenario's process_action.

        Args:
            defender_actions: (num_envs, n_defenders, action_dim)
                             Continuous actions for each defender.

        Returns:
            obs: (num_envs, n_defenders, obs_dim)
            rewards: (num_envs, n_defenders) — per-defender rewards
            done: (num_envs,) — global done flag (bool)
            info: dict with extra logging info
            positions: (num_envs, n_defenders, 2) — for adj matrix construction
        """
        # Build full action list for all agents (defenders + intruders)
        # Intruders get dummy actions — process_action overrides them
        all_actions = []
        defender_action_idx = 0

        for i in range(len(self.env.agents)):
            if i in self.defender_indices:
                # Map from defender index to position in defender_actions
                local_idx = self.defender_indices.index(i)
                all_actions.append(defender_actions[:, local_idx])
            else:
                # Intruder: provide zero action (will be overwritten by script)
                all_actions.append(
                    torch.zeros(self.num_envs, 2, device=self.device)
                )

        # Step VMAS
        all_obs, all_rewards, dones, all_infos = self.env.step(all_actions)

        # Extract defender observations
        defender_obs = torch.stack(
            [all_obs[i] for i in self.defender_indices], dim=1
        )

        # Extract defender rewards
        defender_rewards = torch.stack(
            [all_rewards[i] for i in self.defender_indices], dim=1
        )  # (num_envs, n_defenders)

        # Positions for adjacency matrix
        positions = defender_obs[:, :, 2:4].clone()

        # Aggregate info
        info = {}
        if len(all_infos) > 0 and self.defender_indices:
            first_def_info = all_infos[self.defender_indices[0]]
            if isinstance(first_def_info, dict):
                info = first_def_info

        return defender_obs, defender_rewards, dones, info, positions

    def build_adj(self, positions: torch.Tensor, r_comm: float) -> torch.Tensor:
        """
        Build adjacency matrices from defender positions.

        This is the CRITICAL function for your GNN ablations.
        The adjacency matrix determines the communication graph —
        only defenders within r_comm of each other get edges.

        IMPORTANT: This builds a SEPARATE adjacency matrix for each
        environment in the batch, since agent positions differ.

        Args:
            positions: (num_envs, n_defenders, 2)
            r_comm: communication radius

        Returns:
            adj: (num_envs, n_defenders, n_defenders) — row-normalized
        """
        # Pairwise distances: (num_envs, n_defenders, n_defenders)
        diff = positions.unsqueeze(2) - positions.unsqueeze(1)
        dist = torch.linalg.vector_norm(diff, dim=-1)

        # Adjacency: 1 if within r_comm (includes self-loops)
        adj = (dist <= r_comm).float()

        # Row-normalize (Kipf & Welling renormalization)
        deg = adj.sum(dim=-1, keepdim=True).clamp(min=1)
        adj = adj / deg

        return adj

    def get_team_reward(self, defender_rewards: torch.Tensor) -> torch.Tensor:
        """
        Compute a single team reward by averaging across defenders.
        Use this for the shared critic in CTDE.

        Args:
            defender_rewards: (num_envs, n_defenders)

        Returns:
            team_reward: (num_envs,)
        """
        return defender_rewards.mean(dim=1)

    @property
    def action_dim(self) -> int:
        """Action dimension for defenders (continuous 2D force)."""
        return 2

    @property
    def n_agents(self) -> int:
        """Number of learned agents (defenders only)."""
        return self.n_defenders


# ─────────────────────────────────────────────────────────────────
# Example: integration with your existing training loop
# ─────────────────────────────────────────────────────────────────
if True:
    print("=" * 60)
    print("Guarded Territory Adapter — Integration Test")
    print("=" * 60)

    adapter = GuardedTerritoryAdapter(
        num_envs=8,
        device="cpu",
        n_scouts=3,
        n_interceptors=3,
        n_intruders=3,
        n_zones=2,
        max_steps=200,
    )

    print(f"\nEnvironment configuration:")
    print(f"  Defenders (learned): {adapter.n_defenders}")
    print(f"    - Scouts: {adapter.n_scouts}")
    print(f"    - Interceptors: {adapter.n_interceptors}")
    print(f"  Intruders (scripted): {adapter.n_intruders}")
    print(f"  Observation dim: {adapter.obs_dim}")
    print(f"  Action dim: {adapter.action_dim}")
    print(f"  Agent types: {adapter.agent_types}")

    # ── Reset ──────────────────────────────────────────────────
    obs, positions = adapter.reset()
    print(f"\nAfter reset:")
    print(f"  obs shape: {obs.shape}")
    print(f"  positions shape: {positions.shape}")

    # ── Build adjacency matrix ─────────────────────────────────
    for r_comm in [0.5, 1.0, 1.5, 2.0]:
        adj = adapter.build_adj(positions, r_comm=r_comm)
        avg_degree = (adj > 0).float().sum(dim=-1).mean().item()
        print(f"  r_comm={r_comm}: adj shape={adj.shape}, avg_degree={avg_degree:.2f}")

    # ── Step with random actions ───────────────────────────────
    print(f"\nRunning 50-step rollout with random actions...")
    total_team_reward = torch.zeros(8)
    n_done = 0

    for step in range(50):
        # Random defender actions: (num_envs, n_defenders, 2)
        actions = torch.randn(8, adapter.n_defenders, 2) * 0.5

        obs, rewards, dones, info, positions = adapter.step(actions)
        team_reward = adapter.get_team_reward(rewards)
        total_team_reward += team_reward

        # Rebuild adj each step (dynamic graph!)
        adj = adapter.build_adj(positions, r_comm=1.5)

        if dones.any():
            n_done += dones.sum().item()

    print(f"  Total team reward: {total_team_reward.mean().item():.3f}")
    print(f"  Episodes completed: {n_done}")
    print(f"  Final obs shape: {obs.shape}")
    print(f"  Final adj shape: {adj.shape}")

    # ── Show how this maps to your existing code ───────────────
    print(f"\n{'=' * 60}")
    print("INTEGRATION WITH YOUR GNN-MAPPO:")
    print("=" * 60)
    print("""
    # In your training loop, replace the PettingZoo env with:

    adapter = GuardedTerritoryAdapter(num_envs=64, device="cuda", ...)

    # Your CommPolicy stays the same, just update dimensions:
    policy = CommPolicy(
        obs_dim=adapter.obs_dim,   # auto-computed
        hidden_dim=64,
        action_dim=adapter.action_dim,  # 2 (continuous)
        F_dim=64, G_dim=64, K_hops=K,
    )

    # Rollout collection:
    obs, positions = adapter.reset()
    adj = adapter.build_adj(positions, r_comm=R_COMM)

    for t in range(ROLLOUT_LENGTH):
        # obs[:, i, :] is agent i's observation
        # adj is (num_envs, n_defenders, n_defenders)
        actions, log_probs, entropy = policy.get_actions(obs, adj)
        obs, rewards, dones, info, positions = adapter.step(actions)
        adj = adapter.build_adj(positions, r_comm=R_COMM)
        # Store (obs, actions, rewards, adj, ...) in rollout buffer

    # PPO update uses stored adj matrices (not current ones!)
    # This is the same principle you already have.
    """)

### Observation encoder

3 linear layers 
Encodes observation from obs_dim to F

In [ ]:
import torch
from torch.distributions import Categorical
import torch.nn.functional as F
import torch.nn as nn

class ObservationEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ObservationEncoder, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

        self._init_weights()

    def _init_weights(self):

        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.orthogonal_(layer.weight, gain=2**0.5)
            nn.init.constant_(layer.bias, 0.0)


    def forward(self, x):

        x = self.fc1(x)
        x = F.relu(x)

        x = self.fc2(x)
        x = F.relu(x)

        x = self.fc3(x)

        return x
    


### Graph Conv Layers

Where the GCN as the communication layer comes in
Performs Feature aggregation for neighboring agents, similar to Message Passing Networks

In [ ]:
class GraphConv(nn.Module):
    def __init__(self, F, G, K):
        super(GraphConv, self).__init__()
        self.F = F
        self.G = G
        self.K = K

        self.weights = nn.ParameterList(
            [nn.Parameter(torch.empty(F, G)) for _ in range(K)]
        )

        for p in self.weights:
            nn.init.xavier_uniform_(p)

    def forward(self, X, S):
        Z = X
        accum = X.new_zeros(*X.shape[:-1], self.G)

        for k in range(self.K):
            accum += torch.matmul(Z, self.weights[k])
            Z = torch.matmul(S, Z)

        return accum


### Action Head 

2 Linear layers as policy head 
Takes in agreggated features from each node/agent and determines action accordingly

In [ ]:
class ActionHead(nn.Module):    
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ActionHead, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self._init_weights()


    def _init_weights(self):
        
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.orthogonal_(self.fc2.weight, gain=0.01)
        nn.init.constant_(self.fc2.bias, 0.0)

    def forward(self, G):

        G = self.fc1(G)
        G = F.relu(G)

        G = self.fc2(G)

        return G

### Comm Policy 

Centralized Comm Policy
Learns weights for each previous network
Centralized Training, then each agent gets a copy 

In [ ]:
from torch.distributions import Normal
class CommPolicy(nn.Module):
    def __init__(self, obs_dim, hidden_dim, action_dim, F, G, K):
        super(CommPolicy, self).__init__()

        self.obsEncoder = ObservationEncoder(obs_dim, hidden_dim, F)
        self.graphConv = GraphConv(F, G, K)

        self.mean_head = ActionHead(G, hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, obs, S):
        device = next(self.parameters()).device

        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=device)
        else:
            obs = obs.to(device=device, dtype=torch.float32)

        if not torch.is_tensor(S):
            S = torch.as_tensor(S, dtype=torch.float32, device=device)
        else:
            S = S.to(device=device, dtype=torch.float32)

        obs_encode = self.obsEncoder(obs)
        agg_feats = self.graphConv(obs_encode, S)
        mean = self.mean_head(agg_feats)
        return mean

    def get_actions(self, obs, S):
        mean = self.forward(obs, S)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)

        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)

        return action, log_prob, entropy

    def evaluate_actions(self, obs, S, actions):
        mean = self.forward(obs, S)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)

        log_prob = dist.log_prob(actions).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return log_prob, entropy, mean


### Centailized Critic 

In [ ]:
class CriticNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, device=None):
        super(CriticNetwork, self).__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.device = device

        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=1.0)

        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.constant_(self.fc3.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        x = self.fc1(obs)
        x = F.relu(x)

        x = self.fc2(x)
        x = F.relu(x)

        x = self.fc3(x)
        return x

### Rollout Buffer 

Collect rollout steps for each batch
Computes advantages

In [ ]:
class GNNRolloutBuffer:
    def __init__(self, gamma, gae_lambda, device):
        self.gamma = gamma
        self.gae_lambda = gae_lambda

        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.adj = []

        self.advantages = None
        self.returns = None

        self.device = device

    def add_timestep(self, obs, actions, rewards, dones, log_probs, values, A):
        self.obs.append(obs)
        self.actions.append(actions)
        self.rewards.append(rewards)
        self.dones.append(dones)
        self.log_probs.append(log_probs)
        self.values.append(values)
        self.adj.append(A)

    def compute_advantages(self, last_values):
        N = len(self.obs[0])
        buffer_size = len(self.obs)
        rewards = torch.stack(self.rewards).to(device=self.device, dtype=torch.float32)
        values = torch.stack(self.values).to(device=self.device, dtype=torch.float32)
        dones = torch.stack(self.dones).to(device=self.device, dtype=torch.float32)

        advantages = torch.zeros((buffer_size, N), dtype=torch.float32, device=self.device)
        last_gae = torch.zeros(len(self.obs[0]), dtype=torch.float32, device=self.device)

        for t in reversed(range(buffer_size)):
            if t == buffer_size - 1:
                if not torch.is_tensor(last_values):
                    next_value = torch.as_tensor(last_values, dtype=torch.float32, device=self.device)
                else:
                    next_value = last_values.to(device=self.device, dtype=torch.float32)
            else:
                next_value = values[t + 1]

            deltas = rewards[t] + self.gamma * (1 - dones[t]) * next_value - values[t]
            advantages[t] = deltas + self.gamma * self.gae_lambda * (1 - dones[t]) * last_gae
            last_gae = advantages[t]

        returns = advantages + values
        self.advantages = advantages
        self.returns = returns

    def get_batches(self, B):
        perm = torch.randperm(len(self.obs), device=self.device)
        batches = perm.split(B)

        obs = torch.stack(self.obs).to(device=self.device, dtype=torch.float32)
        actions = torch.stack(self.actions).to(device=self.device, dtype=torch.long)
        log_probs = torch.stack(self.log_probs).to(device=self.device, dtype=torch.float32)
        adj = torch.stack(self.adj).to(device=self.device, dtype=torch.float32)
        adv_mean = self.advantages.mean()
        adv_std = self.advantages.std() + 1e-8
        advantages = (self.advantages - adv_mean) / adv_std

        for idx in batches:
            m_obs = obs[idx]
            m_actions = actions[idx]
            m_log_probs = log_probs[idx]
            m_advantages = advantages[idx]
            m_returns = self.returns[idx]
            m_adj = adj[idx]

            yield m_obs, m_actions, m_log_probs, m_advantages, m_returns, m_adj

    def clear(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.adj = []

        self.advantages = None
        self.returns = None


### Full Trainer Class

In [ ]:
class GNNTrainer:
    def __init__(
        self,
        num_agents,
        env,
        obs_dim,
        hidden_dim,
        action_dim,
        r_comm,
        F,
        G,
        K,
        lr,
        gamma,
        gae_lambda,
        clip_eps,
        value_coef,
        entropy_coef,
        device,
        graphbuilder: GraphBuilder = None
    ):
        self.device = device
        self.num_agents = num_agents
        self.agent_ids = sorted(env.possible_agents)

        self.clip_eps = clip_eps
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef

        self.comm_policy = CommPolicy(
            obs_dim=obs_dim, hidden_dim=hidden_dim, action_dim=action_dim, F=F, G=G, K=K
        ).to(self.device)
        self.comm_optim = Adam(self.comm_policy.parameters(), lr=lr)

        self.critic = CriticNetwork(obs_dim=obs_dim, hidden_dim=hidden_dim, device=self.device)
        self.critic_optim = Adam(self.critic.parameters(), lr=lr)

        self.buffer = GNNRolloutBuffer(gamma=gamma, gae_lambda=gae_lambda, device=self.device)
        self.env = env
        self._running_episode_return = 0.0
        self.metrics_history = {
            "policy_loss": [],
            "value_loss": [],
            "entropy": [],
            "mean_bellman_error": [],
            "mean_episode_return": [],
            "mean_episode_rewards": [],
        }

        obs, info = self.env.reset()
        self.current_obs = torch.stack([torch.from_numpy(obs[a]) for a in self.agent_ids]).to(
            device=self.device, dtype=torch.float32
        )

        self.graph = graphbuilder
        self.r_comm = r_comm

    def _safe_mean(self, values):
        if not values:
            return 0.0
        return float(sum(values) / len(values))

    def collect_rollouts(self, num_steps):
        obs_tensor = self.current_obs
        step_mean_rewards = []
        completed_episode_returns = []
        r_comm = self.r_comm

        for _ in range(num_steps):
            S = self.graph(self.env)

            actions, log_probs, entropy = self.comm_policy.get_actions(obs=obs_tensor, S=S)

            values = self.critic(obs_tensor).detach().squeeze()
            actions_pz = {}
            for i, a_id in enumerate(self.agent_ids):
                actions_pz[a_id] = actions[i].cpu().item()

            next_obs, rewards, dones, truncs, infos = self.env.step(actions_pz)

            rewards_tensor = torch.tensor(
                [rewards[a] for a in self.agent_ids], dtype=torch.float32, device=self.device
            )
            dones_tensor = torch.tensor(
                [dones[a] for a in self.agent_ids], dtype=torch.float32, device=self.device
            )
            self.buffer.add_timestep(
                obs=obs_tensor.detach(),
                actions=actions.detach(),
                rewards=rewards_tensor,
                dones=dones_tensor,
                log_probs=log_probs.detach(),
                values=values,
                A=S.detach(),
            )

            step_mean_rewards.append(rewards_tensor.mean().item())
            self._running_episode_return += rewards_tensor.mean().item()

            if all(dones.values()) or all(truncs.values()):
                completed_episode_returns.append(self._running_episode_return)
                self._running_episode_return = 0.0
                obs, info = self.env.reset()

                obs_tensor = torch.stack([torch.from_numpy(obs[a]) for a in self.agent_ids]).to(
                    device=self.device, dtype=torch.float32
                )
            else:
                obs_tensor = torch.stack([torch.from_numpy(next_obs[a]) for a in self.agent_ids]).to(
                    device=self.device, dtype=torch.float32
                )

            self.current_obs = obs_tensor

        rollout_metrics = {
            "mean_episode_return": self._safe_mean(completed_episode_returns)
            if completed_episode_returns
            else float(self._running_episode_return),
            "mean_episode_rewards": self._safe_mean(step_mean_rewards),
        }
        self.metrics_history["mean_episode_return"].append(rollout_metrics["mean_episode_return"])
        self.metrics_history["mean_episode_rewards"].append(rollout_metrics["mean_episode_rewards"])

        return obs_tensor, rollout_metrics

    def update(self, last_obs, num_epochs=30, B=64):
        with torch.no_grad():
            last_obs_tensor = last_obs.to(device=self.device, dtype=torch.float32)
            last_values = self.critic(last_obs_tensor).squeeze(-1)
        self.buffer.compute_advantages(last_values=last_values)

        policy_losses = []
        entropies = []
        value_losses = []
        bellman_errors = []

        for _ in range(num_epochs):
            for (obs, actions, old_log_probs, advantages, returns, A) in self.buffer.get_batches(B):
                new_lp, entropy, _ = self.comm_policy.evaluate_actions(obs, A, actions)

                ratio = torch.exp(new_lp - old_log_probs)
                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * advantages

                policy_loss = -torch.min(surr1, surr2).mean()
                entropy_loss = entropy.mean()
                loss = policy_loss - self.entropy_coef * entropy_loss

                self.comm_optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.comm_policy.parameters(), max_norm=0.5)
                self.comm_optim.step()

                policy_losses.append(policy_loss.item())
                entropies.append(entropy_loss.item())

                B_size, N, obs_dim = obs.shape
                flat_obs = obs.reshape(B_size * N, obs_dim)
                flat_returns = returns.reshape(B_size * N)

                pred_values = self.critic(flat_obs).squeeze(-1)
                td_error = flat_returns - pred_values
                value_loss = F.mse_loss(pred_values, flat_returns)
                mean_bellman_error = td_error.abs().mean()

                self.critic_optim.zero_grad()
                value_loss.backward()
                nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5)
                self.critic_optim.step()

                value_losses.append(value_loss.item())
                bellman_errors.append(mean_bellman_error.item())

        self.buffer.clear()
        update_metrics = {
            "policy_loss": self._safe_mean(policy_losses),
            "value_loss": self._safe_mean(value_losses),
            "entropy": self._safe_mean(entropies),
            "mean_bellman_error": self._safe_mean(bellman_errors),
        }
        self.metrics_history["policy_loss"].append(update_metrics["policy_loss"])
        self.metrics_history["value_loss"].append(update_metrics["value_loss"])
        self.metrics_history["entropy"].append(update_metrics["entropy"])
        self.metrics_history["mean_bellman_error"].append(update_metrics["mean_bellman_error"])

        return update_metrics


### Experiment Sweep: K and r_comm
Runs a full grid over K in [2, 3, 5] and r_comm in [1.0, 1.5, 2.0] with TOTAL_TIMESTEPS = 200_000 and ROLLOUT_LENGTH = 2048.

In [ ]:
import matplotlib.pyplot as plt
import os

NUM_AGENTS = 5
MAX_CYCLES = 100

action_dim = 5
F_dim = 64
G_dim = 64
hidden_dim = 64

lr = 3e-4
gamma = 0.99
gae_lambda = 0.85
clip_eps = 0.2
value_coef = 0.5
entropy_coef = 0.01

TOTAL_TIMESTEPS = 200_000
ROLLOUT_LENGTH = 2048
BATCH_SIZE = 64
NUM_EPOCHS = 10

K_VALUES = [2, 3, 5]
R_COMM_VALUES = [1.0, 1.5, 2.0]
SEED = 42
LOG_EVERY = 10



OUTPUT_DIR = "outputs/experiments_k_r"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"

print(f"Using device: {device}")
print(f"Sweep settings: K={K_VALUES}, r_comm={R_COMM_VALUES}")
print(f"TOTAL_TIMESTEPS={TOTAL_TIMESTEPS}, ROLLOUT_LENGTH={ROLLOUT_LENGTH}")

metrics_to_plot = [
    "policy_loss",
    "value_loss",
    "entropy",
    "mean_bellman_error",
    "mean_episode_return",
    "mean_episode_rewards",
]


def plot_metrics(metrics_history, title, save_path):
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    axes = axes.flatten()

    for i, metric_name in enumerate(metrics_to_plot):
        values = metrics_history.get(metric_name, [])
        x = range(1, len(values) + 1)
        axes[i].plot(x, values, linewidth=1.8)
        axes[i].set_title(metric_name)
        axes[i].set_xlabel("Iteration")
        axes[i].set_ylabel(metric_name)
        axes[i].grid(True, alpha=0.3)

    fig.suptitle(title)
    plt.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


results = {}
summary_rows = []

for K_hops in K_VALUES:
    for R_COMM in R_COMM_VALUES:
        config_name = f"K={K_hops}, r_comm={R_COMM}"
        print()
        print(f"=== Running {config_name} ===")

        torch.manual_seed(SEED)
        np.random.seed(SEED)

        env = parallel_env(N=NUM_AGENTS, max_cycles=MAX_CYCLES)
        graph_builder = GraphBuilder(build_method="dist", norm_method="raw", r_comm=R_COMM, device=device)
        try:
            obs, info = env.reset(seed=SEED)
        except TypeError:
            obs, info = env.reset()

        obs_dim = obs["agent_0"].shape[0]

        trainer = GNNTrainer(
            num_agents=NUM_AGENTS,
            env=env,
            obs_dim=obs_dim,
            hidden_dim=hidden_dim,
            action_dim=action_dim,
            F=F_dim,
            G=G_dim,
            K=K_hops,
            lr=lr,
            gamma=gamma,
            gae_lambda=gae_lambda,
            clip_eps=clip_eps,
            value_coef=value_coef,
            entropy_coef=entropy_coef,
            device=device,
            graphbuilder=graph_builder,
            r_comm=R_COMM,
        )

        steps_done = 0
        iteration = 0

        while steps_done < TOTAL_TIMESTEPS:
            rollout_steps = min(ROLLOUT_LENGTH, TOTAL_TIMESTEPS - steps_done)
            last_obs, rollout_metrics = trainer.collect_rollouts(num_steps=rollout_steps)
            update_metrics = trainer.update(last_obs, num_epochs=NUM_EPOCHS, B=BATCH_SIZE)

            steps_done += rollout_steps
            iteration += 1

            should_log = (
                iteration == 1
                or iteration % LOG_EVERY == 0
                or steps_done == TOTAL_TIMESTEPS
            )
            if should_log:
                print(
                    f"[{config_name}] Iter {iteration} | "
                    f"steps={steps_done}/{TOTAL_TIMESTEPS} | "
                    f"policy_loss={update_metrics['policy_loss']:.4f} | "
                    f"value_loss={update_metrics['value_loss']:.4f} | "
                    f"entropy={update_metrics['entropy']:.4f} | "
                    f"bellman={update_metrics['mean_bellman_error']:.4f} | "
                    f"ep_return={rollout_metrics['mean_episode_return']:.4f} | "
                    f"ep_reward={rollout_metrics['mean_episode_rewards']:.4f}"
                )

        results[(K_hops, R_COMM)] = trainer.metrics_history

        final_return = trainer.metrics_history["mean_episode_return"][-1]
        final_reward = trainer.metrics_history["mean_episode_rewards"][-1]
        summary_rows.append((K_hops, R_COMM, final_return, final_reward))

        plot_path = os.path.join(OUTPUT_DIR, f"metrics_K{K_hops}_r{R_COMM:.1f}.png")
        plot_metrics(
            trainer.metrics_history,
            title=f"Training Metrics ({config_name})",
            save_path=plot_path,
        )
        print(f"Saved metrics plot: {plot_path}")

        env.close()


# Summary heatmap of final mean episode return
return_matrix = np.full((len(K_VALUES), len(R_COMM_VALUES)), np.nan, dtype=np.float32)
for K_hops, R_COMM, final_return, _ in summary_rows:
    i = K_VALUES.index(K_hops)
    j = R_COMM_VALUES.index(R_COMM)
    return_matrix[i, j] = final_return

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(return_matrix, cmap="viridis")
ax.set_xticks(range(len(R_COMM_VALUES)))
ax.set_yticks(range(len(K_VALUES)))
ax.set_xticklabels([str(v) for v in R_COMM_VALUES])
ax.set_yticklabels([str(v) for v in K_VALUES])
ax.set_xlabel("r_comm")
ax.set_ylabel("K")
ax.set_title("Final Mean Episode Return")

for i in range(len(K_VALUES)):
    for j in range(len(R_COMM_VALUES)):
        ax.text(j, i, f"{return_matrix[i, j]:.2f}", ha="center", va="center", color="white")

fig.colorbar(im, ax=ax)
plt.tight_layout()
summary_path = os.path.join(OUTPUT_DIR, "summary_final_return_heatmap.png")
plt.savefig(summary_path, dpi=180)
plt.show()
print(f"Saved summary heatmap: {summary_path}")

print()
print("Final summary (K, r_comm, final_return, final_reward):")
for row in summary_rows:
    print(row)
